# Noisy FitzHugh-Nagumo: coherence from noise

FitzHugh-Nagumo in its excitable regime driven by white noise (`wiener w`, integrated
by Euler-Maruyama). Noise alone triggers spikes; at moderate noise they are surprisingly
regular. Each run uses new random numbers: press *run again*.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.1 and its AUTO through pyauto-xpp.*

In [1]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [2]:
mb = (ModelBuilder("noisy_fhn")
      .wiener("w")
      .ode("v", "(v-v^3/3-u)/eps")
      .ode("u", "v+a+sig*w")
      .parameter(eps=0.05, a=1.05, sig=0.2)
      .initial(v=-1.05, u=-0.664)
      .option(meth="euler", dt=0.002, total=60))
print(mb.render())

wiener w
v'=(v-v^3/3-u)/eps
u'=v+a+sig*w
par eps=0.05,a=1.05,sig=0.2
init v=-1.05,u=-0.664
@ meth=euler,dt=0.002,total=60
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [3]:
ex = Explorer(mb, kind="plane", x="v", y="u", window=(-2.5, 2.5, -1, 1.5), df=False,
              sliders={"sig": (0, 0.6), "a": (0.8, 1.3)}, title="Noisy FitzHugh-Nagumo")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Regularity of the spike train against noise**: the coefficient of variation of the
intervals (low = regular) has a minimum.

In [4]:
import numpy as np
S = np.linspace(0.05, 0.6, 12)
cv = []
for s in S:
    v = ex.m.run(keep=False, sig=s, total=400)["v"]; t = ex.m.data.t
    sp = t[1:][(v[:-1] < 0) & (v[1:] >= 0)]
    d = np.diff(sp)
    cv.append(d.std() / d.mean() if len(d) > 3 else np.nan)
show_figure(go.Figure(go.Scatter(x=S, y=cv, mode="lines+markers")), xaxis_title="sig", yaxis_title="CV of intervals", height=300)

## Things to try

- Set `sig` to 0: no spikes at all (the rest state is stable).